# AIR Blackbox — EU AI Act Compliance LLM Fine-Tune (v12)

Fine-tunes **Llama-3.1-8B-Instruct** on **8,874** balanced EU AI Act compliance examples using **Unsloth + LoRA**.

**What's new in v12**: Balanced PASS/FAIL training data (52% PASS vs old 22%). Adds real code patterns from Haystack, LangChain, CrewAI with specific function/class citations. Uses Alpaca format with `sample_context` and `total_files` so the model learns not to assume everything is missing.

**Runtime**: Colab Pro (A100 GPU) or free T4

**IMPORTANT — Read before running:**
- Run steps **in order**, one at a time
- Step 6 (training) takes **~4 hours** on A100 — checkpoints save to **Google Drive** every 500 steps
- **CRASH-PROOF**: Even if Colab fully disconnects, checkpoints survive on Google Drive
- If Colab disconnects mid-training, re-run Steps 1-5, then run Step 6b to recover
- Use Runtime > Change runtime type > **A100 GPU**

---

## Step 1: Install Dependencies

This installs Unsloth (fast LoRA training), TRL (training library), and other deps. Takes ~2 min.

In [ ]:
%%capture
# Install Unsloth + compatible TRL version
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.9.0" peft accelerate bitsandbytes datasets

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Verify installations
import trl
print(f"TRL version: {trl.__version__}")

import torch
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")
print("Step 1 complete!")

## Step 2: Upload Training Data

Upload `training_data_combined_v4.jsonl` (8,874 examples, ~40MB).

**Click the file picker** that appears when you run this cell, then select the file from your Mac.

If you get a "file not found" error later, re-run this cell to re-upload (Colab deletes files on runtime restart).

In [ ]:
from google.colab import files
import os

DATA_FILE = 'training_data_combined_v4.jsonl'

if not os.path.exists(DATA_FILE):
    print(f"Please upload {DATA_FILE}...")
    uploaded = files.upload()
    # Handle if they uploaded with a different name
    uploaded_name = list(uploaded.keys())[0]
    if uploaded_name != DATA_FILE:
        os.rename(uploaded_name, DATA_FILE)
    print(f"Uploaded: {DATA_FILE} ({os.path.getsize(DATA_FILE) / 1024 / 1024:.1f} MB)")
else:
    print(f"Training data already present: {DATA_FILE} ({os.path.getsize(DATA_FILE) / 1024 / 1024:.1f} MB)")

# Quick validation
import json
with open(DATA_FILE) as f:
    first_line = json.loads(f.readline())
    line_count = 1 + sum(1 for _ in f)

# Check PASS/FAIL balance
pass_count = 0
fail_count = 0
with open(DATA_FILE) as f:
    for line in f:
        output = str(json.loads(line).get('output', ''))
        pass_count += output.count('**: PASS')
        fail_count += output.count('**: FAIL')

print(f"Examples: {line_count:,}")
print(f"Fields: {list(first_line.keys())}")
print(f"PASS/FAIL balance: {100*pass_count/(pass_count+fail_count):.0f}% PASS / {100*fail_count/(pass_count+fail_count):.0f}% FAIL")
print("Step 2 complete!")

## Step 3: Prepare Training Data

Converts the JSONL into Alpaca-format prompts that the model learns from.

In [ ]:
import json
from datasets import Dataset

DATA_FILE = 'training_data_combined_v4.jsonl'

# Load JSONL
examples = []
with open(DATA_FILE, 'r') as f:
    for line in f:
        row = json.loads(line.strip())
        output = row.get('output', '')
        if isinstance(output, dict):
            output = json.dumps(output, indent=2)
        examples.append({
            'instruction': row.get('instruction', ''),
            'input': row.get('input', ''),
            'output': str(output),
        })

print(f"Loaded {len(examples):,} training examples")

# Format as Alpaca prompts
def format_alpaca(ex):
    if ex['input']:
        return f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{ex['instruction']}

### Input:
{ex['input']}

### Response:
{ex['output']}"""
    else:
        return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{ex['instruction']}

### Response:
{ex['output']}"""

dataset = Dataset.from_list([{'text': format_alpaca(ex)} for ex in examples])
print(f"Dataset ready: {len(dataset):,} examples")
print(f"\nSample (first 300 chars):\n{dataset[0]['text'][:300]}...")
print("\nStep 3 complete!")

## Step 4: Load Model (4-bit quantized Llama 3.1 8B)

Downloads and loads the base model. Takes ~3 min on first run.

**`max_seq_length=4096`** matches the Ollama model's context window. Use 2048 on T4 if you get OOM errors.

In [ ]:
import torch
torch.cuda.empty_cache()  # Clear any leftover GPU memory

from unsloth import FastLanguageModel

# Use 4096 on A100, reduce to 2048 on T4 if you get OOM errors
max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print(f"Model loaded! max_seq_length={max_seq_length}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print("Step 4 complete!")

## Step 5: Apply LoRA Adapters

Adds trainable LoRA layers on top of the frozen base model. Only ~1.7% of parameters are trained — this is what makes it fast and memory-efficient.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      # LoRA rank (higher = more capacity, more VRAM)
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                     'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',  # Saves ~40% VRAM
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print(f'GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB')
print('Step 5 complete!')

## Step 6: Train the Model + AUTO-SAVE TO GOOGLE DRIVE

This cell trains AND saves in one go. All checkpoints go to **Google Drive** so they survive runtime disconnects.

**Crash-proof features:**
- Mounts Google Drive FIRST
- Saves checkpoints every **500 steps** to Google Drive (survives full disconnect)
- Auto-saves LoRA adapter to Google Drive immediately after training completes
- Auto-downloads the adapter zip file

**If runtime disconnects mid-training:** Re-run Steps 1-5, then run Step 6b to recover from Google Drive checkpoint.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BULLETPROOF TRAINING — verifies Drive is writable BEFORE training
# Saves checkpoints every 200 steps + custom callback as backup
# ═══════════════════════════════════════════════════════════════════

# ── STEP A: Mount Google Drive and PROVE it works ──
from google.colab import drive
import os, time, json

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/air-blackbox-training"
os.makedirs(SAVE_DIR, exist_ok=True)

# WRITE A TEST FILE — if this fails, Drive is broken and we stop
test_file = f"{SAVE_DIR}/_drive_write_test_{int(time.time())}.txt"
try:
    with open(test_file, 'w') as f:
        f.write(f"Drive write test at {time.time()}")
    # Read it back to confirm
    with open(test_file, 'r') as f:
        content = f.read()
    os.remove(test_file)
    print(f"✅ Google Drive is WRITABLE at {SAVE_DIR}")
except Exception as e:
    raise RuntimeError(
        f"❌ STOPPING — Google Drive is NOT writable!\n"
        f"Error: {e}\n"
        f"Fix: Runtime > Disconnect and delete runtime, then start fresh.\n"
        f"Do NOT proceed — training will be lost."
    )

# ── STEP B: Custom callback that force-saves to Drive every 200 steps ──
import torch
from transformers import TrainerCallback
import shutil

class GoogleDriveSaveCallback(TrainerCallback):
    """Force-saves model to Google Drive as a backup, independent of HF checkpoints."""

    def __init__(self, save_dir, model_ref, tokenizer_ref, save_every_n_steps=200):
        self.save_dir = save_dir
        self.model_ref = model_ref
        self.tokenizer_ref = tokenizer_ref
        self.save_every_n_steps = save_every_n_steps
        self.last_save_step = 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        step = state.global_step
        if step > 0 and step - self.last_save_step >= self.save_every_n_steps:
            backup_dir = f"{self.save_dir}/backup-step-{step}"
            try:
                self.model_ref.save_pretrained(backup_dir)
                self.tokenizer_ref.save_pretrained(backup_dir)

                # Verify the save actually worked
                if os.path.exists(f"{backup_dir}/adapter_config.json"):
                    print(f"\n💾 BACKUP SAVED to Google Drive: step {step}")
                    self.last_save_step = step

                    # Clean up old backups (keep last 3)
                    import glob
                    backups = sorted(
                        glob.glob(f"{self.save_dir}/backup-step-*"),
                        key=lambda x: int(x.split("-")[-1])
                    )
                    for old in backups[:-3]:
                        shutil.rmtree(old, ignore_errors=True)
                else:
                    print(f"\n⚠️ Backup at step {step} — adapter_config.json missing!")
            except Exception as e:
                print(f"\n⚠️ Backup save failed at step {step}: {e}")

# ── STEP C: Set up trainer with BOTH HF checkpoints and custom backups ──
torch.cuda.empty_cache()

from trl import SFTTrainer
from transformers import TrainingArguments

drive_callback = GoogleDriveSaveCallback(
    save_dir=SAVE_DIR,
    model_ref=model,
    tokenizer_ref=tokenizer,
    save_every_n_steps=200
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=16,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        warmup_steps=100,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        logging_steps=50,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=3407,
        output_dir=SAVE_DIR,
        report_to='none',
        save_strategy='steps',
        save_steps=500,
        save_total_limit=3,
    ),
    callbacks=[drive_callback],
)

# ── STEP D: Final pre-flight check ──
print(f"\nTraining config:")
print(f"  Dataset: {len(dataset):,} examples")
print(f"  Epochs: 3")
print(f"  Batch size: 1 x 16 accumulation = effective 16")
print(f"  Max seq length: {max_seq_length}")
print(f"  Precision: bfloat16")
print(f"  GPU memory before training: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"")
print(f"  💾 HF checkpoints: every 500 steps → {SAVE_DIR}")
print(f"  💾 Backup saves:   every 200 steps → {SAVE_DIR}/backup-step-*")
print(f"  ✅ Google Drive verified writable")
print(f"")
print(f"Starting training...")
print(f"---")

stats = trainer.train()

# ── STEP E: Training complete — save final model ──
print(f"\n{'='*50}")
print(f"TRAINING COMPLETE")
print(f"Final loss: {stats.training_loss:.4f}")
print(f"Runtime: {stats.metrics.get('train_runtime', 0) / 60:.1f} minutes")
print(f"{'='*50}")

LORA_DIR = f"{SAVE_DIR}/air-compliance-lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

# Verify the final save
if os.path.exists(f"{LORA_DIR}/adapter_config.json"):
    size_mb = sum(
        os.path.getsize(os.path.join(LORA_DIR, f))
        for f in os.listdir(LORA_DIR)
        if os.path.isfile(os.path.join(LORA_DIR, f))
    ) / 1024 / 1024
    print(f"\n✅ FINAL MODEL SAVED to Google Drive: {LORA_DIR} ({size_mb:.1f} MB)")
else:
    print(f"\n❌ WARNING: Final save may have failed — check {LORA_DIR}")

# Also save locally and download
model.save_pretrained('./air-compliance-lora')
tokenizer.save_pretrained('./air-compliance-lora')

shutil.make_archive('air-compliance-lora', 'zip', './air-compliance-lora')
print("Zipped: air-compliance-lora.zip")

from google.colab import files
files.download('air-compliance-lora.zip')
print("Download triggered! Check your browser downloads.")
print("\n🎉 Step 6 COMPLETE — model trained AND saved to Google Drive!")

## Step 6b: Recovery from Google Drive (ONLY if runtime crashed mid-training)

Skip this cell if Step 6 completed successfully. Only use if the runtime disconnected during training.

**Re-run Steps 1-5 first**, then run this cell. It will find your checkpoints on Google Drive and recover the model.

In [ ]:
# ═══ RECOVERY FROM GOOGLE DRIVE — only run if runtime crashed ═══
# Re-run Steps 1-5 first, then run this cell

from google.colab import drive
drive.mount('/content/drive')

import os, glob

SAVE_DIR = "/content/drive/MyDrive/air-blackbox-training"

print(f"Scanning Google Drive for saved models...\n")

# Check for completed model first
LORA_DIR = f"{SAVE_DIR}/air-compliance-lora"
if os.path.exists(LORA_DIR) and os.path.exists(f"{LORA_DIR}/adapter_config.json"):
    print(f"✅ Found COMPLETED model at {LORA_DIR}")
    files_list = os.listdir(LORA_DIR)
    size_mb = sum(os.path.getsize(os.path.join(LORA_DIR, f)) for f in files_list if os.path.isfile(os.path.join(LORA_DIR, f))) / 1024 / 1024
    print(f"   Files: {len(files_list)}, Size: {size_mb:.1f} MB")
    print("   Downloading...")
    import shutil
    from google.colab import files
    if os.path.exists('./air-compliance-lora'):
        shutil.rmtree('./air-compliance-lora')
    shutil.copytree(LORA_DIR, './air-compliance-lora')
    shutil.make_archive('air-compliance-lora', 'zip', './air-compliance-lora')
    files.download('air-compliance-lora.zip')
    print("   Download triggered!")

else:
    # Look for BOTH types of saves: HF checkpoints and custom backups
    hf_checkpoints = sorted(
        glob.glob(f"{SAVE_DIR}/checkpoint-*"),
        key=lambda x: int(x.split("-")[-1])
    )
    backup_saves = sorted(
        glob.glob(f"{SAVE_DIR}/backup-step-*"),
        key=lambda x: int(x.split("-")[-1])
    )

    print(f"HF checkpoints found: {len(hf_checkpoints)}")
    for cp in hf_checkpoints:
        step = cp.split("-")[-1]
        has_config = os.path.exists(f"{cp}/adapter_config.json")
        print(f"  checkpoint-{step} {'✅' if has_config else '❌ (incomplete)'}")

    print(f"\nBackup saves found: {len(backup_saves)}")
    for bs in backup_saves:
        step = bs.split("-")[-1]
        has_config = os.path.exists(f"{bs}/adapter_config.json")
        print(f"  backup-step-{step} {'✅' if has_config else '❌ (incomplete)'}")

    # Find the latest VALID save (has adapter_config.json)
    all_saves = hf_checkpoints + backup_saves
    valid_saves = [s for s in all_saves if os.path.exists(f"{s}/adapter_config.json")]

    if valid_saves:
        # Sort by step number
        latest = max(valid_saves, key=lambda x: int(x.rstrip('/').split("-")[-1]))
        step = latest.rstrip('/').split("-")[-1]
        print(f"\n🔄 Recovering from latest valid save: step {step}")
        print(f"   Path: {latest}")

        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=latest,
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=True,
        )
        print("   Model recovered from checkpoint!")

        # Save as final LoRA adapter
        model.save_pretrained(LORA_DIR)
        tokenizer.save_pretrained(LORA_DIR)
        model.save_pretrained('./air-compliance-lora')
        tokenizer.save_pretrained('./air-compliance-lora')
        print(f"   LoRA adapter saved to Google Drive AND locally")

        import shutil
        from google.colab import files
        shutil.make_archive('air-compliance-lora', 'zip', './air-compliance-lora')
        files.download('air-compliance-lora.zip')
        print("   Download triggered!")
    else:
        print(f"\n❌ No valid saves found anywhere in {SAVE_DIR}")
        if os.path.exists(SAVE_DIR):
            contents = os.listdir(SAVE_DIR)
            print(f"   Directory contents: {contents if contents else '(empty)'}")
        else:
            print("   Directory doesn't exist")
        print("\n   You need to retrain from scratch — re-run Step 6")

## Step 7: Test the Model

Quick sanity check — give it some code and see if it generates a real compliance analysis.

In [ ]:
FastLanguageModel.for_inference(model)

test_code = """import logging
from typing import Any, Dict, List, Optional

logger = logging.getLogger(__name__)

class AgentExecutor:
    \"\"\"Executes agent actions with safety controls.\"\"\"

    def __init__(self, agent, tools: List, max_iterations: int = 25):
        self.agent = agent
        self.tools = tools
        self.max_iterations = max_iterations
        self._iteration = 0

    def invoke(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        \"\"\"Run agent with execution boundary and error handling.\"\"\"
        self._iteration += 1
        if self._iteration > self.max_iterations:
            logger.warning("Max iterations exceeded: %d", self.max_iterations)
            raise RuntimeError(f"Agent exceeded {self.max_iterations} iterations")

        try:
            result = self.agent.run(input_data)
            logger.info("Agent completed iteration %d", self._iteration)
            return result
        except Exception as e:
            logger.error("Agent error at iteration %d: %s", self._iteration, e)
            return {"error": str(e), "fallback": True}"""

prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze this Python code for EU AI Act compliance. This is a targeted sample of 3 compliance-relevant source files from a project with 120 Python files. Assess ONLY what is visible in the code below — do not assume patterns are missing if they could exist in files not shown.

For each of Articles 9, 10, 11, 12, 14, and 15: report status (pass if evidence found, warn if partial, fail only if clearly absent), cite specific evidence from the code (function names, patterns, line references), and give fix recommendations. Output as a JSON array.

### Input:
{test_code}

### Response:
"""

inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.1, do_sample=True)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = result.split('### Response:')[-1].strip()

print('=== MODEL OUTPUT ===')
print(response)
print('\nStep 7 complete!')

## Step 8: Save & Export

Saves three versions:
1. **LoRA adapter** (~50MB) — lightweight, loads on top of base model
2. **Merged model** (~16GB) — full standalone model
3. **GGUF** — for running locally with Ollama

In [ ]:
# Save LoRA adapter (small, fast)
model.save_pretrained('./air-compliance-lora')
tokenizer.save_pretrained('./air-compliance-lora')
print('LoRA adapter saved! (~50MB)')

# Save merged full model
model.save_pretrained_merged('./air-compliance-model', tokenizer, save_method='merged_16bit')
print('Merged model saved!')

print('\nStep 8a complete!')

In [ ]:
# GGUF export for Ollama (optional — takes a few minutes)
try:
    model.save_pretrained_gguf(
        './air-compliance-gguf',
        tokenizer,
        quantization_method=['q4_k_m', 'q8_0'],
    )
    print('GGUF models saved! You can run these with Ollama.')
except Exception as e:
    print(f'GGUF export failed (this is OK): {e}')
    print('You can quantize locally with llama.cpp after downloading the merged model.')

print('\nStep 8b complete!')

In [ ]:
# Download the LoRA adapter to your computer
from google.colab import files
import shutil

shutil.make_archive('air-compliance-lora', 'zip', './air-compliance-lora')
files.download('air-compliance-lora.zip')
print('Downloading LoRA adapter...')

---

## Done!

Your EU AI Act compliance LLM is trained on **8,874 balanced examples** covering:

- **Balanced PASS/FAIL**: 52% PASS / 48% FAIL (vs old 22%/78%)
- **Real code patterns**: Haystack, LangChain, CrewAI with specific function/class citations
- **Alpaca format**: Uses `sample_context` and `total_files` — model learns not to assume everything is missing
- **Every relevant EU AI Act article**: 9 (Risk), 10 (Data), 11 (Docs), 12 (Records), 14 (Human Oversight), 15 (Security)
- 8+ AI frameworks (LangChain, CrewAI, OpenAI, Anthropic, AutoGen, Haystack, LlamaIndex, Semantic Kernel)

**Next steps:**
1. Download the GGUF file
2. Copy to `~/models/air-compliance/` and rebuild: `ollama create air-compliance -f Modelfile`
3. Test: `air-blackbox comply --scan ~/Desktop/haystack-test -v`
4. Push to registry: `ollama push airblackbox/air-compliance`

In [ ]:
# Uncomment these lines to push to HuggingFace:

# !huggingface-cli login --token YOUR_HF_TOKEN_HERE
# HUB_MODEL = 'airblackbox/air-compliance-llama3.1-8b'
# model.push_to_hub_merged(HUB_MODEL, tokenizer, save_method='merged_16bit')
# print(f'Pushed to: https://huggingface.co/{HUB_MODEL}')